# 🔍 Receipt Forgery Detection — Full Training Pipeline
### SROIE Dataset → Forgery Generator → CNN Classifier → U-Net Segmenter

```
SROIE Receipts (Kaggle)
        ↓
Forgery Generator  ← reads OCR .txt coords, patches price fields
        ↓
Preprocessing      ← augmentation, normalisation
        ↓
   ┌────┴────┐
CNN Classifier   U-Net Segmenter
   └────┬────┘
        ↓
  Save models → use in Streamlit app
```

**Outputs (save these for Streamlit):**
- `classifier_final.pth`
- `unet_final.pth`
- `anomaly_model.pkl`
- `config.json`

> ⚠️ Runtime → Change runtime type → **T4 GPU** before running!


## 📦 Section 1: Install Dependencies

In [1]:
%%capture
!pip install kaggle timm grad-cam albumentations segmentation-models-pytorch
!pip install pytesseract opencv-python-headless scikit-learn matplotlib seaborn
!apt-get install -y tesseract-ocr
print('✅ Done!')

## 🔧 Section 2: Imports & Config

In [2]:
import os, cv2, re, json, random, shutil, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
import pytesseract

CFG = {
    'data_dir'      : '/content/sroie',
    'output_dir'    : '/content/outputs',
    'ckpt_dir'      : '/content/checkpoints',
    'backbone'      : 'efficientnet_b3',
    'img_size'      : 224,
    'batch_size'    : 16,
    'clf_epochs'    : 25,
    'seg_epochs'    : 15,
    'lr'            : 2e-4,
    'weight_decay'  : 1e-5,
    'patience'      : 6,
    'val_split'     : 0.15,
    'test_split'    : 0.10,
    'use_amp'       : True,
    'seed'          : 42,
    # Forgery generator settings
    'forgery_types' : ['price_replace', 'date_replace', 'total_replace'],
    'forgery_ratio' : 1.0,   # 1.0 = equal real/forged
    # Anomaly detection
    'max_valid_amount'    : 10000.0,
    'if_contamination'    : 0.15,
}

def seed_all(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG['seed'])

for d in [CFG['output_dir'], CFG['ckpt_dir']]:
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print('✅ Config ready!')

Device : cuda
GPU    : Tesla T4
✅ Config ready!


## 📥 Section 3: Download SROIE from Kaggle

In [3]:
# ── Step 1: Upload your kaggle.json API key ────────────────────────
# Get it from: kaggle.com → Account → API → Create New Token
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()   # select kaggle.json

# Setup Kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# ── Step 2: Download SROIE ─────────────────────────────────────────
!kaggle datasets download -d urbikn/sroie-datasetv2 -p /content/
!unzip -q /content/sroie-datasetv2.zip -d /content/sroie_raw
print('✅ SROIE downloaded!')

Upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/urbikn/sroie-datasetv2
License(s): other
100% 834M/834M [00:04<00:00, 214MB/s]

✅ SROIE downloaded!


In [4]:
# ── Step 3: Explore downloaded structure ──────────────────────────
import os
for root, dirs, files_list in os.walk('/content/sroie_raw'):
    level = root.replace('/content/sroie_raw', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files_list[:5]:
            print(f'{indent}  {f}')
        if len(files_list) > 5:
            print(f'{indent}  ... ({len(files_list)} files total)')

sroie_raw/
  SROIE2019/
    test/
      entities/
      img/
      box/
    layoutlm-base-uncased/
    train/
      entities/
      img/
      box/


In [5]:
# ── Step 4: Parse SROIE into a unified DataFrame ──────────────────
def parse_sroie(raw_dir):
    """
    SROIE structure (typical):
        train/
            img/  *.jpg
            box/  *.txt  ← OCR boxes: x1,y1,x2,y1,x2,y2,x1,y2,text
            entities/ *.txt ← key fields: TOTAL, DATE, etc.
        test/
            img/  *.jpg
            box/  *.txt
    Adapts automatically to flat or nested layouts.
    """
    rows = []
    raw = Path(raw_dir)

    # Find all jpg images recursively
    img_paths = list(raw.rglob('*.jpg')) + list(raw.rglob('*.JPG'))
    print(f'Found {len(img_paths)} images')

    for ip in img_paths:
        stem = ip.stem
        # Try to find matching OCR box file
        box_candidates = [
            ip.parent.parent / 'box' / f'{stem}.txt',
            ip.parent.parent / 'boxes' / f'{stem}.txt',
            ip.with_suffix('.txt'),
            ip.parent / f'{stem}.txt',
        ]
        box_file = next((p for p in box_candidates if p.exists()), None)

        # Try to find entity file (has TOTAL, DATE)
        ent_candidates = [
            ip.parent.parent / 'entities' / f'{stem}.txt',
            ip.parent.parent / 'entity' / f'{stem}.txt',
        ]
        ent_file = next((p for p in ent_candidates if p.exists()), None)

        rows.append({
            'image_path' : str(ip),
            'stem'       : stem,
            'box_file'   : str(box_file) if box_file else None,
            'ent_file'   : str(ent_file) if ent_file else None,
        })

    df = pd.DataFrame(rows)
    has_box = df['box_file'].notna().sum()
    has_ent = df['ent_file'].notna().sum()
    print(f'  With OCR box files  : {has_box}/{len(df)}')
    print(f'  With entity files   : {has_ent}/{len(df)}')
    return df

df_sroie = parse_sroie('/content/sroie_raw')
print('\nSample rows:')
print(df_sroie.head(3).to_string())

Found 973 images
  With OCR box files  : 973/973
  With entity files   : 973/973

Sample rows:
                                               image_path          stem                                                box_file                                                     ent_file
0  /content/sroie_raw/SROIE2019/test/img/X51005447841.jpg  X51005447841  /content/sroie_raw/SROIE2019/test/box/X51005447841.txt  /content/sroie_raw/SROIE2019/test/entities/X51005447841.txt
1  /content/sroie_raw/SROIE2019/test/img/X51006556808.jpg  X51006556808  /content/sroie_raw/SROIE2019/test/box/X51006556808.txt  /content/sroie_raw/SROIE2019/test/entities/X51006556808.txt
2  /content/sroie_raw/SROIE2019/test/img/X51007339119.jpg  X51007339119  /content/sroie_raw/SROIE2019/test/box/X51007339119.txt  /content/sroie_raw/SROIE2019/test/entities/X51007339119.txt


## 🔨 Section 4: Forgery Generator

In [6]:
"""
improved_forgery_generator.py
==============================
Drop-in replacement for generate_forgery() in the Colab notebook.
Same function signature and return values — no pipeline changes needed.

Improvements over original:
  1. Local patch sampling instead of flat color fill
  2. Multiple font styles + misalignment simulation
  3. Partial number tampering (e.g. 100 → 180)
  4. JPEG compression artifact simulation
  5. Noise + texture injection
  6. Copy-move forgery type
  7. Smudge/blur tampering type
  8. Soft mask edges (Gaussian blur on mask)
  9. Randomised number of tampered regions
  10. Annotation noise on mask boundaries

Usage:
  Replace generate_forgery() in Section 4 of the Colab notebook with this file's version.
  All other code stays identical.
"""

import cv2
import re
import random
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont
import io


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 1: Realistic background fill (local patch sampling)
# ─────────────────────────────────────────────────────────────────────────────
def _sample_background_patch(img_np, x1, y1, x2, y2):
    """
    Instead of using a flat average colour, sample a patch from a nearby
    clean region of the image and use it as the fill texture.
    This prevents the tell-tale flat rectangle visible in naive forgeries.
    """
    H, W = img_np.shape[:2]
    bh = max(y2 - y1, 8)
    bw = max(x2 - x1, 8)

    # Try sampling from directly above or below the target box
    candidates = []
    if y1 - bh - 5 >= 0:
        candidates.append(img_np[y1 - bh - 5 : y1 - 5, x1:x2])
    if y2 + bh + 5 < H:
        candidates.append(img_np[y2 + 5 : y2 + bh + 5, x1:x2])
    # Fallback: sample from a random clean area far from the box
    if not candidates:
        ry = random.randint(0, max(0, H - bh - 1))
        candidates.append(img_np[ry : ry + bh, x1:x2])

    patch = random.choice(candidates)

    # Resize patch to exactly fill the target box
    if patch.shape[0] == 0 or patch.shape[1] == 0:
        # Last resort: flat colour from local mean
        region = img_np[max(0, y1-5):y2+5, max(0, x1-5):x2+5]
        colour = region.mean(axis=(0, 1)).astype(np.uint8) if region.size > 0 else np.array([240, 240, 240])
        patch = np.full((bh, bw, 3), colour, dtype=np.uint8)
    else:
        patch = cv2.resize(patch, (bw, bh))

    return patch


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 2: Noise injection into a region
# ─────────────────────────────────────────────────────────────────────────────
def _inject_noise(region, intensity=8):
    """
    Add subtle Gaussian noise to a region so it matches surrounding texture.
    Prevents the fill from looking too clean vs. the noisy scanned background.
    """
    noise = np.random.normal(0, intensity, region.shape).astype(np.int16)
    noisy = np.clip(region.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return noisy


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 3: JPEG compression artifact simulation
# ─────────────────────────────────────────────────────────────────────────────
def _simulate_jpeg_artifacts(img_np, quality=None):
    """
    Encode to JPEG at low quality and decode back.
    Creates the block artefacts seen in scanned/photographed receipts.
    Applied to the whole image at the end.
    """
    if quality is None:
        quality = random.randint(60, 85)
    pil_img = Image.fromarray(img_np)
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return np.array(Image.open(buf).convert('RGB'))


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 4: Realistic text rendering with misalignment
# ─────────────────────────────────────────────────────────────────────────────
def _render_text_realistic(draw, text, x1, y1, x2, y2, img_np):
    """
    Renders replacement text with:
      - Random ink colour variation (slightly off-black)
      - Slight x/y position jitter
      - Random font size within box height
      - Optional slight rotation via a temp image
    """
    bh = max(y2 - y1, 10)
    bw = max(x2 - x1, 10)

    # Ink colour: dark but not perfect black — simulates faded/uneven ink
    ink_base = random.randint(15, 60)
    ink_variation = random.randint(-15, 15)
    ink_r = max(0, min(255, ink_base + ink_variation + random.randint(-10, 10)))
    ink_g = max(0, min(255, ink_base + random.randint(-5, 5)))
    ink_b = max(0, min(255, ink_base + random.randint(-5, 5)))
    ink_colour = (ink_r, ink_g, ink_b)

    # Font size: fit within box height with some randomness
    font_size = max(8, int(bh * random.uniform(0.55, 0.85)))

    # Position jitter: ±3px misalignment
    jitter_x = random.randint(-3, 3)
    jitter_y = random.randint(-2, 2)
    tx = x1 + 2 + jitter_x
    ty = y1 + max(0, (bh - font_size) // 2) + jitter_y

    # Try loading a system font, fall back to default
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except:
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
        except:
            font = ImageFont.load_default()

    # Draw with optional slight opacity variation (simulated via colour)
    if random.random() < 0.3:
        # Faded ink — lighten the text colour
        fade = random.randint(40, 80)
        ink_colour = tuple(min(255, c + fade) for c in ink_colour)

    draw.text((tx, ty), text, fill=ink_colour, font=font)


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 5: Partial number tampering
# ─────────────────────────────────────────────────────────────────────────────
def _partial_number_tamper(original_text):
    """
    Instead of replacing the whole number, modify only 1–2 digits.
    E.g.  '125.50'  →  '185.50'  or  '925.50'
    Makes forgery harder to detect visually.
    """
    digits = list(original_text)
    digit_positions = [i for i, c in enumerate(digits) if c.isdigit()]

    if not digit_positions:
        return _fake_amount_full(original_text)

    # Choose 1–2 digit positions to modify
    n_changes = random.randint(1, min(2, len(digit_positions)))
    chosen = random.sample(digit_positions, n_changes)

    for pos in chosen:
        original_digit = int(digits[pos])
        # Ensure the new digit is meaningfully different (+1 to +5 or +6 to +9)
        delta = random.choice([1, 2, 3, 4, 5, 6, 7, 8, 9])
        new_digit = (original_digit + delta) % 10
        # Avoid 0 as first digit of a number
        if pos == digit_positions[0] and new_digit == 0:
            new_digit = random.randint(1, 9)
        digits[pos] = str(new_digit)

    return ''.join(digits)


def _fake_amount_full(original):
    """Full replacement: inflate by 5x–50x."""
    digits = re.sub(r'[^\d]', '', original)
    if not digits:
        digits = '100'
    val = float(digits) / 100.0
    multiplier = random.choice([5, 8, 10, 15, 20, 25, 50])
    return f'{val * multiplier:,.2f}'


def _fake_date(original):
    """Replace year with a plausible alternative year."""
    year = random.choice(['2019', '2020', '2021', '2022', '2025', '2026'])
    result = re.sub(r'\d{4}', year, original, count=1)
    return result if result != original else original + '*'


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 6: Soft mask generation
# ─────────────────────────────────────────────────────────────────────────────
def _make_soft_mask(H, W, tampered_boxes, annotation_noise=True):
    """
    Generates a binary mask with soft edges and slight boundary noise.
    - Gaussian blur on mask edges (not hard rectangles)
    - Small random expand/shrink to simulate annotation imprecision
    """
    mask = np.zeros((H, W), dtype=np.float32)

    for box in tampered_boxes:
        x1, y1, x2, y2 = box['bbox']
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)

        # Annotation noise: randomly expand or shrink boundary by 0–4px
        if annotation_noise:
            pad = random.randint(-2, 4)
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(W, x2 + pad)
            y2 = min(H, y2 + pad)

        mask[y1:y2, x1:x2] = 1.0

    # Soft edges: apply Gaussian blur to mask, then re-threshold
    # This makes boundaries fuzzy rather than sharp rectangles
    if mask.max() > 0:
        blur_radius = random.choice([3, 5, 7])
        mask_blurred = cv2.GaussianBlur(mask, (blur_radius, blur_radius), sigmaX=1.5)
        # Keep mostly binary but with soft transition at edges
        mask_final = np.where(mask_blurred > 0.3, 1.0, mask_blurred / 0.3 * 0.5)
        # Convert to uint8 for saving (>0.5 → 255)
        return (mask_final > 0.5).astype(np.uint8)

    return mask.astype(np.uint8)


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 7: Smudge/blur tampering type (new)
# ─────────────────────────────────────────────────────────────────────────────
def _apply_smudge(img_np, x1, y1, x2, y2):
    """
    Smudge forgery: heavily blur the target region without replacing text.
    Makes the original value unreadable — simulates physical smudging.
    """
    roi = img_np[y1:y2, x1:x2].copy()
    # Strong motion blur
    kernel_size = random.choice([9, 11, 15])
    kernel = np.zeros((kernel_size, kernel_size))
    if random.random() < 0.5:
        kernel[kernel_size // 2, :] = 1  # horizontal
    else:
        kernel[:, kernel_size // 2] = 1  # vertical
    kernel /= kernel_size
    blurred = cv2.filter2D(roi, -1, kernel)
    # Add noise on top
    blurred = _inject_noise(blurred, intensity=12)
    return blurred


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 8: Copy-move forgery type (new)
# ─────────────────────────────────────────────────────────────────────────────
def _apply_copy_move(img_np, x1, y1, x2, y2):
    """
    Copy-move forgery: copy a patch from elsewhere in the image
    and paste it over the target region.
    Simulates someone hiding a value by covering it with similar-looking content.
    """
    H, W = img_np.shape[:2]
    bh = max(y2 - y1, 8)
    bw = max(x2 - x1, 8)

    # Find a source region away from the target
    attempts = 0
    while attempts < 10:
        src_x = random.randint(0, max(0, W - bw - 1))
        src_y = random.randint(0, max(0, H - bh - 1))
        # Make sure source doesn't overlap with target
        if abs(src_x - x1) > bw * 2 or abs(src_y - y1) > bh * 2:
            break
        attempts += 1

    src_patch = img_np[src_y:src_y + bh, src_x:src_x + bw].copy()
    if src_patch.shape[:2] != (bh, bw):
        src_patch = cv2.resize(src_patch, (bw, bh))

    # Slight colour adjustment to make it look like it belongs
    adjustment = random.randint(-10, 10)
    src_patch = np.clip(src_patch.astype(np.int16) + adjustment, 0, 255).astype(np.uint8)

    return src_patch


# ─────────────────────────────────────────────────────────────────────────────
# EXISTING HELPERS (kept from original, used as fallback)
# ─────────────────────────────────────────────────────────────────────────────
AMOUNT_RE = re.compile(r'^\$?\d{1,6}(?:[,.]?\d{2,3})*(?:\.\d{2})?$')
DATE_RE   = re.compile(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}')
TOTAL_RE  = re.compile(r'(?:total|amount|grand|subtotal)', re.I)

def classify_box(box):
    t = box['text'].strip()
    if TOTAL_RE.search(t):  return 'total_label'
    if DATE_RE.search(t):   return 'date'
    if AMOUNT_RE.match(t):  return 'amount'
    return 'other'

def parse_ocr_boxes(box_file):
    boxes = []
    try:
        with open(box_file, encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                parts = line.split(',')
                if len(parts) < 9: continue
                try:
                    coords = [int(x) for x in parts[:8]]
                    text   = ','.join(parts[8:]).strip()
                    x1 = min(coords[0],coords[2],coords[4],coords[6])
                    y1 = min(coords[1],coords[3],coords[5],coords[7])
                    x2 = max(coords[0],coords[2],coords[4],coords[6])
                    y2 = max(coords[1],coords[3],coords[5],coords[7])
                    boxes.append({'bbox': [x1,y1,x2,y2], 'text': text})
                except:
                    continue
    except:
        pass
    return boxes


# ─────────────────────────────────────────────────────────────────────────────
# MAIN: Improved generate_forgery()
# Same signature as original — drop-in replacement
# ─────────────────────────────────────────────────────────────────────────────
def generate_forgery(image_path, box_file, forgery_type='price_replace'):
    """
    Improved forgery generator.

    Args:
        image_path  : path to receipt image
        box_file    : path to SROIE OCR .txt file (or None)
        forgery_type: 'price_replace' | 'date_replace' | 'total_replace'
                      (extended internally with smudge/copy_move/partial variants)

    Returns:
        (forged_image_np, binary_mask_np, tampered_boxes)
        Identical return format to original generate_forgery().
    """
    # ── Load image ────────────────────────────────────────────────
    img = cv2.imread(str(image_path))
    if img is None:
        return None, None, []
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    # ── Parse OCR boxes ───────────────────────────────────────────
    boxes = parse_ocr_boxes(box_file) if box_file else []
    if not boxes:
        # Fallback: tamper the typical total region (bottom-right)
        boxes = [{'bbox': [W//2, int(H*0.75), W-5, H-5], 'text': '99.99'}]

    for b in boxes:
        b['tag'] = classify_box(b)

    # ── Internally expand forgery_type to include new variants ────
    # With 30% chance, randomly pick smudge or copy_move instead
    effective_type = forgery_type
    rand_roll = random.random()
    if rand_roll < 0.15:
        effective_type = 'smudge'
    elif rand_roll < 0.25:
        effective_type = 'copy_move'
    elif rand_roll < 0.40:
        effective_type = 'partial_replace'  # partial digit tampering

    # ── Select target boxes ───────────────────────────────────────
    if forgery_type in ('price_replace', 'partial_replace'):
        targets = [b for b in boxes if b['tag'] == 'amount']
    elif forgery_type == 'date_replace':
        targets = [b for b in boxes if b['tag'] == 'date']
    elif forgery_type == 'total_replace':
        targets = []
        for i, b in enumerate(boxes):
            if b['tag'] == 'total_label' and i + 1 < len(boxes):
                targets.append(boxes[i + 1])
        if not targets:
            targets = [b for b in boxes if b['tag'] == 'amount'][-1:]
    elif effective_type in ('smudge', 'copy_move'):
        # Any text region works for smudge/copy-move
        targets = [b for b in boxes if b['tag'] in ('amount', 'date', 'total_label')]
    else:
        targets = [b for b in boxes if b['tag'] == 'amount']

    if not targets:
        return None, None, []

    # ── Randomise number of tampered regions (1–3) ────────────────
    max_tamper = min(len(targets), random.randint(1, 3))
    selected   = random.sample(targets, max_tamper)

    # ── Build forged image ────────────────────────────────────────
    forged     = img.copy()
    pil_img    = Image.fromarray(forged)
    draw       = ImageDraw.Draw(pil_img)
    tampered   = []

    for b in selected:
        x1, y1, x2, y2 = b['bbox']
        # Clamp to image bounds
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        if x2 <= x1 or y2 <= y1:
            continue

        arr = np.array(pil_img)   # work on latest state

        # ── Choose tampering technique based on effective_type ────
        if effective_type == 'smudge':
            # ── Technique A: Smudge (blur + noise) ───────────────
            smudged = _apply_smudge(arr, x1, y1, x2, y2)
            arr[y1:y2, x1:x2] = smudged
            replacement = '[smudged]'

        elif effective_type == 'copy_move':
            # ── Technique B: Copy-move patch ─────────────────────
            patch = _apply_copy_move(arr, x1, y1, x2, y2)
            arr[y1:y2, x1:x2] = patch
            replacement = '[copy-move]'

        else:
            # ── Technique C: Text replacement (improved) ──────────

            # Step 1: Fill with realistic background patch
            bg_patch = _sample_background_patch(arr, x1, y1, x2, y2)
            bg_patch  = _inject_noise(bg_patch, intensity=random.randint(4, 10))
            arr[y1:y2, x1:x2] = bg_patch

            # Step 2: Determine replacement text
            if effective_type == 'partial_replace':
                replacement = _partial_number_tamper(b['text'])
            elif b['tag'] == 'date':
                replacement = _fake_date(b['text'])
            else:
                # 50% chance of partial, 50% full replacement
                if random.random() < 0.5:
                    replacement = _partial_number_tamper(b['text'])
                else:
                    replacement = _fake_amount_full(b['text'])

            # Step 3: Render text realistically
            pil_img = Image.fromarray(arr)
            draw    = ImageDraw.Draw(pil_img)
            _render_text_realistic(draw, replacement, x1, y1, x2, y2, arr)
            arr = np.array(pil_img)

            # Step 4: Optionally add a subtle partial overwrite effect
            # (faint smear at top of box to simulate pen stroke)
            if random.random() < 0.25:
                smear_h = max(2, (y2 - y1) // 4)
                smear   = arr[y1:y1 + smear_h, x1:x2].copy()
                smear   = cv2.GaussianBlur(smear, (5, 3), sigmaX=2)
                smear   = _inject_noise(smear, intensity=6)
                arr[y1:y1 + smear_h, x1:x2] = smear

        pil_img = Image.fromarray(arr)
        draw    = ImageDraw.Draw(pil_img)

        tampered.append({
            'bbox'        : [x1, y1, x2, y2],
            'original'    : b['text'],
            'replacement' : replacement,
            'technique'   : effective_type,
        })

    # ── Global post-processing ────────────────────────────────────
    final_arr = np.array(pil_img)

    # 1. Subtle overall noise (simulates scanner grain)
    if random.random() < 0.6:
        grain = np.random.normal(0, random.uniform(2, 6), final_arr.shape)
        final_arr = np.clip(final_arr.astype(np.float32) + grain, 0, 255).astype(np.uint8)

    # 2. JPEG compression artifacts (simulates re-saved receipt scan)
    if random.random() < 0.5:
        final_arr = _simulate_jpeg_artifacts(final_arr, quality=random.randint(65, 88))

    # 3. Slight brightness/contrast variation (simulates uneven scan lighting)
    if random.random() < 0.4:
        alpha = random.uniform(0.92, 1.08)   # contrast
        beta  = random.randint(-8, 8)         # brightness
        final_arr = np.clip(final_arr.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

    # ── Generate soft mask ────────────────────────────────────────
    mask = _make_soft_mask(H, W, tampered, annotation_noise=True)

    return final_arr, mask, tampered


# ─────────────────────────────────────────────────────────────────────────────
# QUICK VISUAL TEST
# Run this cell standalone in Colab to verify improvements:
#
#   from improved_forgery_generator import generate_forgery
#   import matplotlib.pyplot as plt, cv2
#
#   orig = cv2.cvtColor(cv2.imread('sample.jpg'), cv2.COLOR_BGR2RGB)
#   forged, mask, changes = generate_forgery('sample.jpg', 'sample.txt', 'price_replace')
#
#   fig, axes = plt.subplots(1, 3, figsize=(16, 6))
#   axes[0].imshow(orig);    axes[0].set_title('Original');      axes[0].axis('off')
#   axes[1].imshow(forged);  axes[1].set_title('Forged');        axes[1].axis('off')
#   axes[2].imshow(mask, cmap='Reds'); axes[2].set_title('Mask'); axes[2].axis('off')
#   plt.show()
#   print(changes)
# ─────────────────────────────────────────────────────────────────────────────

In [7]:
# ── Build full forged dataset ─────────────────────────────────────

import cv2
import re
import random
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont
import io

# ─────────────────────────────────────────────────────────────────────────────
# HELPER 1: Realistic background fill (local patch sampling) - FIX APPLIED
# ─────────────────────────────────────────────────────────────────────────────
def _sample_background_patch(img_np, x1, y1, x2, y2):
    """
    Instead of using a flat average colour, sample a patch from a nearby
    clean region of the image and use it as the fill texture.
    This prevents the tell-tale flat rectangle visible in naive forgeries.
    """
    H, W = img_np.shape[:2]
    bh = max(y2 - y1, 8)
    bw = max(x2 - x1, 8)

    # Try sampling from directly above or below the target box
    candidates = []
    if y1 - bh - 5 >= 0:
        candidates.append(img_np[y1 - bh - 5 : y1 - 5, x1:x2])
    if y2 + bh + 5 < H:
        candidates.append(img_np[y2 + 5 : y2 + bh + 5, x1:x2])
    # Fallback: sample from a random clean area far from the box
    if not candidates:
        ry = random.randint(0, max(0, H - bh - 1))
        candidates.append(img_np[ry : ry + bh, x1:x2])

    patch = random.choice(candidates)

    # Resize patch to exactly fill the target box
    target_width = x2 - x1
    target_height = y2 - y1

    if patch.shape[0] == 0 or patch.shape[1] == 0:
        # Last resort: flat colour from local mean
        region = img_np[max(0, y1-5):y2+5, max(0, x1-5):x2+5]
        colour = region.mean(axis=(0, 1)).astype(np.uint8) if region.size > 0 else np.array([240, 240, 240])
        patch = np.full((target_height, target_width, 3), colour, dtype=np.uint8)
    else:
        patch = cv2.resize(patch, (target_width, target_height))

    return patch


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 2: Noise injection into a region (Copied for generate_forgery context)
# ─────────────────────────────────────────────────────────────────────────────
def _inject_noise(region, intensity=8):
    """
    Add subtle Gaussian noise to a region so it matches surrounding texture.
    Prevents the fill from looking too clean vs. the noisy scanned background.
    """
    noise = np.random.normal(0, intensity, region.shape).astype(np.int16)
    noisy = np.clip(region.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return noisy


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 3: JPEG compression artifact simulation (Copied for generate_forgery context)
# ─────────────────────────────────────────────────────────────────────────────
def _simulate_jpeg_artifacts(img_np, quality=None):
    """
    Encode to JPEG at low quality and decode back.
    Creates the block artefacts seen in scanned/photographed receipts.
    Applied to the whole image at the end.
    """
    if quality is None:
        quality = random.randint(60, 85)
    pil_img = Image.fromarray(img_np)
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return np.array(Image.open(buf).convert('RGB'))


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 4: Realistic text rendering with misalignment (Copied for generate_forgery context)
# ─────────────────────────────────────────────────────────────────────────────
def _render_text_realistic(draw, text, x1, y1, x2, y2, img_np):
    """
    Renders replacement text with:
      - Random ink colour variation (slightly off-black)
      - Slight x/y position jitter
      - Random font size within box height
      - Optional slight rotation via a temp image
    """
    bh = max(y2 - y1, 10)
    bw = max(x2 - x1, 10)

    # Ink colour: dark but not perfect black — simulates faded/uneven ink
    ink_base = random.randint(15, 60)
    ink_variation = random.randint(-15, 15)
    ink_r = max(0, min(255, ink_base + ink_variation + random.randint(-10, 10)))
    ink_g = max(0, min(255, ink_base + random.randint(-5, 5)))
    ink_b = max(0, min(255, ink_base + random.randint(-5, 5)))
    ink_colour = (ink_r, ink_g, ink_b)

    # Font size: fit within box height with some randomness
    font_size = max(8, int(bh * random.uniform(0.55, 0.85)))

    # Position jitter: ±3px misalignment
    jitter_x = random.randint(-3, 3)
    jitter_y = random.randint(-2, 2)
    tx = x1 + 2 + jitter_x
    ty = y1 + max(0, (bh - font_size) // 2) + jitter_y

    # Try loading a system font, fall back to default
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except:
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
        except:
            font = ImageFont.load_default()

    # Draw with optional slight opacity variation (simulated via colour)
    if random.random() < 0.3:
        # Faded ink — lighten the text colour
        fade = random.randint(40, 80)
        ink_colour = tuple(min(255, c + fade) for c in ink_colour)

    draw.text((tx, ty), text, fill=ink_colour, font=font)


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 5: Partial number tampering (Copied for generate_forgery context)
# ─────────────────────────────────────────────────────────────────────────────
def _partial_number_tamper(original_text):
    """
    Instead of replacing the whole number, modify only 1–2 digits.
    E.g.  '125.50'  →  '185.50'  or  '925.50'
    Makes forgery harder to detect visually.
    """
    digits = list(original_text)
    digit_positions = [i for i, c in enumerate(digits) if c.isdigit()]

    if not digit_positions:
        return _fake_amount_full(original_text)

    # Choose 1–2 digit positions to modify
    n_changes = random.randint(1, min(2, len(digit_positions)))
    chosen = random.sample(digit_positions, n_changes)

    for pos in chosen:
        original_digit = int(digits[pos])
        # Ensure the new digit is meaningfully different (+1 to +5 or +6 to +9)
        delta = random.choice([1, 2, 3, 4, 5, 6, 7, 8, 9])
        new_digit = (original_digit + delta) % 10
        # Avoid 0 as first digit of a number
        if pos == digit_positions[0] and new_digit == 0:
            new_digit = random.randint(1, 9)
        digits[pos] = str(new_digit)

    return ''.join(digits)


def _fake_amount_full(original):
    """Full replacement: inflate by 5x–50x."""
    digits = re.sub(r'[^\d]', '', original)
    if not digits:
        digits = '100'
    val = float(digits) / 100.0
    multiplier = random.choice([5, 8, 10, 15, 20, 25, 50])
    return f'{val * multiplier:,.2f}'


def _fake_date(original):
    """Replace year with a plausible alternative year."""
    year = random.choice(['2019', '2020', '2021', '2022', '2025', '2026'])
    result = re.sub(r'\d{4}', year, original, count=1)
    return result if result != original else original + '*'


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 6: Soft mask generation (Copied for generate_forgery context)
# ─────────────────────────────────────────────────────────────────────────────
def _make_soft_mask(H, W, tampered_boxes, annotation_noise=True):
    """
    Generates a binary mask with soft edges and slight boundary noise.
    - Gaussian blur on mask edges (not hard rectangles)
    - Small random expand/shrink to simulate annotation imprecision
    """
    mask = np.zeros((H, W), dtype=np.float32)

    for box in tampered_boxes:
        x1, y1, x2, y2 = box['bbox']
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)

        # Annotation noise: randomly expand or shrink boundary by 0–4px
        if annotation_noise:
            pad = random.randint(-2, 4)
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(W, x2 + pad)
            y2 = min(H, H + pad)

        mask[y1:y2, x1:x2] = 1.0

    # Soft edges: apply Gaussian blur to mask, then re-threshold
    # This makes boundaries fuzzy rather than sharp rectangles
    if mask.max() > 0:
        blur_radius = random.choice([3, 5, 7])
        mask_blurred = cv2.GaussianBlur(mask, (blur_radius, blur_radius), sigmaX=1.5)
        # Keep mostly binary but with soft transition at edges
        mask_final = np.where(mask_blurred > 0.3, 1.0, mask_blurred / 0.3 * 0.5)
        # Convert to uint8 for saving (>0.5 → 255)
        return (mask_final > 0.5).astype(np.uint8)

    return mask.astype(np.uint8)


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 7: Smudge/blur tampering type (new) (Copied for generate_forgery context)
# ─────────────────────────────────────────────────────────────────────────────
def _apply_smudge(img_np, x1, y1, x2, y2):
    """
    Smudge forgery: heavily blur the target region without replacing text.
    Makes the original value unreadable — simulates physical smudging.
    """
    roi = img_np[y1:y2, x1:x2].copy()
    # Strong motion blur
    kernel_size = random.choice([9, 11, 15])
    kernel = np.zeros((kernel_size, kernel_size))
    if random.random() < 0.5:
        kernel[kernel_size // 2, :] = 1  # horizontal
    else:
        kernel[:, kernel_size // 2] = 1  # vertical
    kernel /= kernel_size
    blurred = cv2.filter2D(roi, -1, kernel)
    # Add noise on top
    blurred = _inject_noise(blurred, intensity=12)
    return blurred


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 8: Copy-move forgery type (new) (Copied for generate_forgery context)
# ─────────────────────────────────────────────────────────────────────────────
def _apply_copy_move(img_np, x1, y1, x2, y2):
    """
    Copy-move forgery: copy a patch from elsewhere in the image
    and paste it over the target region.
    """
    H, W = img_np.shape[:2]

    # Use ACTUAL target dimensions (clamped), not bh/bw
    target_h = y2 - y1   # ← fix: was using bh which could differ
    target_w = x2 - x1   # ← fix: was using bw which could differ

    if target_h <= 0 or target_w <= 0:
        return img_np[y1:y2, x1:x2].copy()

    # Find a source region away from the target
    attempts = 0
    src_x, src_y = 0, 0
    while attempts < 10:
        src_x = random.randint(0, max(0, W - target_w - 1))
        src_y = random.randint(0, max(0, H - target_h - 1))
        if abs(src_x - x1) > target_w * 2 or abs(src_y - y1) > target_h * 2:
            break
        attempts += 1

    src_patch = img_np[src_y:src_y + target_h, src_x:src_x + target_w].copy()

    # ← fix: always resize to exact target shape regardless of source
    if src_patch.shape[0] != target_h or src_patch.shape[1] != target_w:
        src_patch = cv2.resize(src_patch, (target_w, target_h))

    # Slight colour adjustment
    adjustment = random.randint(-10, 10)
    src_patch = np.clip(src_patch.astype(np.int16) + adjustment, 0, 255).astype(np.uint8)

    return src_patch


# ─────────────────────────────────────────────────────────────────────────────
# EXISTING HELPERS (kept from original, used as fallback) (Copied for generate_forgery context)
# ─────────────────────────────────────────────────────────────────────────────
AMOUNT_RE = re.compile(r'^\$?\d{1,6}(?:[,.]?\d{2,3})*(?:\.\d{2})?$')
DATE_RE   = re.compile(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}')
TOTAL_RE  = re.compile(r'(?:total|amount|grand|subtotal)', re.I)

def classify_box(box):
    t = box['text'].strip()
    if TOTAL_RE.search(t):  return 'total_label'
    if DATE_RE.search(t):   return 'date'
    if AMOUNT_RE.match(t):  return 'amount'
    return 'other'

def parse_ocr_boxes(box_file):
    boxes = []
    try:
        with open(box_file, encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                parts = line.split(',')
                if len(parts) < 9: continue
                try:
                    coords = [int(x) for x in parts[:8]]
                    text   = ','.join(parts[8:]).strip()
                    x1 = min(coords[0],coords[2],coords[4],coords[6])
                    y1 = min(coords[1],coords[3],coords[5],coords[7])
                    x2 = max(coords[0],coords[2],coords[4],coords[6])
                    y2 = max(coords[1],coords[3],coords[5],coords[7])
                    boxes.append({'bbox': [x1,y1,x2,y2], 'text': text})
                except:
                    continue
    except:
        pass
    return boxes


# ─────────────────────────────────────────────────────────────────────────────
# MAIN: Improved generate_forgery() - Redefined to use corrected helper
# Same signature as original — drop-in replacement
# ─────────────────────────────────────────────────────────────────────────────
def generate_forgery(image_path, box_file, forgery_type='price_replace'):
    """
    Improved forgery generator.

    Args:
        image_path  : path to receipt image
        box_file    : path to SROIE OCR .txt file (or None)
        forgery_type: 'price_replace' | 'date_replace' | 'total_replace'
                      (extended internally with smudge/copy_move/partial variants)

    Returns:
        (forged_image_np, binary_mask_np, tampered_boxes)
        Identical return format to original generate_forgery().
    """
    # ── Load image ────────────────────────────────────────────────
    img = cv2.imread(str(image_path))
    if img is None:
        return None, None, []
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    # ── Parse OCR boxes ───────────────────────────────────────────
    boxes = parse_ocr_boxes(box_file) if box_file else []
    if not boxes:
        # Fallback: tamper the typical total region (bottom-right)
        boxes = [{'bbox': [W//2, int(H*0.75), W-5, H-5], 'text': '99.99'}]

    for b in boxes:
        b['tag'] = classify_box(b)

    # ── Internally expand forgery_type to include new variants ────
    # With 30% chance, randomly pick smudge or copy_move instead
    effective_type = forgery_type
    rand_roll = random.random()
    if rand_roll < 0.15:
        effective_type = 'smudge'
    elif rand_roll < 0.25:
        effective_type = 'copy_move'
    elif rand_roll < 0.40:
        effective_type = 'partial_replace'  # partial digit tampering

    # ── Select target boxes ───────────────────────────────────────
    if forgery_type in ('price_replace', 'partial_replace'):
        targets = [b for b in boxes if b['tag'] == 'amount']
    elif forgery_type == 'date_replace':
        targets = [b for b in boxes if b['tag'] == 'date']
    elif forgery_type == 'total_replace':
        targets = []
        for i, b in enumerate(boxes):
            if b['tag'] == 'total_label' and i + 1 < len(boxes):
                targets.append(boxes[i + 1])
        if not targets:
            targets = [b for b in boxes if b['tag'] == 'amount'][-1:]
    elif effective_type in ('smudge', 'copy_move'):
        # Any text region works for smudge/copy-move
        targets = [b for b in boxes if b['tag'] in ('amount', 'date', 'total_label')]
    else:
        targets = [b for b in boxes if b['tag'] == 'amount']

    if not targets:
        return None, None, []

    # ── Randomise number of tampered regions (1–3) ────────────────
    max_tamper = min(len(targets), random.randint(1, 3))
    selected   = random.sample(targets, max_tamper)

    # ── Build forged image ────────────────────────────────────────
    forged     = img.copy()
    pil_img    = Image.fromarray(forged)
    draw       = ImageDraw.Draw(pil_img)
    tampered   = []

    for b in selected:
        x1, y1, x2, y2 = b['bbox']
        # Clamp to image bounds
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        if x2 <= x1 or y2 <= y1:
            continue

        arr = np.array(pil_img)   # work on latest state

        # ── Choose tampering technique based on effective_type ────
        if effective_type == 'smudge':
            # ── Technique A: Smudge (blur + noise) ───────────────
            smudged = _apply_smudge(arr, x1, y1, x2, y2)
            arr[y1:y2, x1:x2] = smudged
            replacement = '[smudged]'

        elif effective_type == 'copy_move':
            # ── Technique B: Copy-move patch ─────────────────────
            patch = _apply_copy_move(arr, x1, y1, x2, y2)
            arr[y1:y2, x1:x2] = patch
            replacement = '[copy-move]'

        else:
            # ── Technique C: Text replacement (improved) ──────────

            # Step 1: Fill with realistic background patch
            bg_patch = _sample_background_patch(arr, x1, y1, x2, y2)
            bg_patch  = _inject_noise(bg_patch, intensity=random.randint(4, 10))
            arr[y1:y2, x1:x2] = bg_patch

            # Step 2: Determine replacement text
            if effective_type == 'partial_replace':
                replacement = _partial_number_tamper(b['text'])
            elif b['tag'] == 'date':
                replacement = _fake_date(b['text'])
            else:
                # 50% chance of partial, 50% full replacement
                if random.random() < 0.5:
                    replacement = _partial_number_tamper(b['text'])
                else:
                    replacement = _fake_amount_full(b['text'])

            # Step 3: Render text realistically
            pil_img = Image.fromarray(arr)
            draw    = ImageDraw.Draw(pil_img)
            _render_text_realistic(draw, replacement, x1, y1, x2, y2, arr)
            arr = np.array(pil_img)

            # Step 4: Optionally add a subtle partial overwrite effect
            # (faint smear at top of box to simulate pen stroke)
            if random.random() < 0.25:
                smear_h = max(2, (y2 - y1) // 4)
                smear   = arr[y1:y1 + smear_h, x1:x2].copy()
                smear   = cv2.GaussianBlur(smear, (5, 3), sigmaX=2)
                smear   = _inject_noise(smear, intensity=6)
                arr[y1:y1 + smear_h, x1:x2] = smear

        pil_img = Image.fromarray(arr)
        draw    = ImageDraw.Draw(pil_img)

        tampered.append({
            'bbox'        : [x1, y1, x2, y2],
            'original'    : b['text'],
            'replacement' : replacement,
            'technique'   : effective_type,
        })

    # ── Global post-processing ────────────────────────────────────
    final_arr = np.array(pil_img)

    # 1. Subtle overall noise (simulates scanner grain)
    if random.random() < 0.6:
        grain = np.random.normal(0, random.uniform(2, 6), final_arr.shape)
        final_arr = np.clip(final_arr.astype(np.float32) + grain, 0, 255).astype(np.uint8)

    # 2. JPEG compression artifacts (simulates re-saved receipt scan)
    if random.random() < 0.5:
        final_arr = _simulate_jpeg_artifacts(final_arr, quality=random.randint(65, 88))

    # 3. Slight brightness/contrast variation (simulates uneven scan lighting)
    if random.random() < 0.4:
        alpha = random.uniform(0.92, 1.08)   # contrast
        beta  = random.randint(-8, 8)         # brightness
        final_arr = np.clip(final_arr.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

    # ── Generate soft mask ────────────────────────────────────────
    mask = _make_soft_mask(H, W, tampered, annotation_noise=True)

    return final_arr, mask, tampered

def build_dataset(df_sroie, data_dir, forgery_ratio=1.0):
    """
    Creates:
      data_dir/real/       ← original SROIE images
      data_dir/forged/     ← generated forged images
      data_dir/masks/      ← binary mask .png per forged image
      data_dir/metadata.csv
    """
    for d in ['real','forged','masks']:
        os.makedirs(os.path.join(data_dir, d), exist_ok=True)

    rows = []
    n_forged_target = int(len(df_sroie) * forgery_ratio)
    forgery_pool = df_sroie[df_sroie['box_file'].notna()].sample(
        min(n_forged_target, len(df_sroie[df_sroie['box_file'].notna()])),
        random_state=CFG['seed']
    )

    # ── Real images ───────────────────────────────────────────────
    print('Copying real images...')
    for _, row in tqdm(df_sroie.iterrows(), total=len(df_sroie)):
        dst = os.path.join(data_dir, 'real', f'{row["stem"]}.jpg')
        shutil.copy(row['image_path'], dst)
        rows.append({
            'image_path' : dst,
            'mask_path'  : None,
            'label'      : 0,
            'label_name' : 'real',
            'forgery_type': None,
            'changes'    : None,
        })

    # ── Forged images ─────────────────────────────────────────────
    print('\nGenerating forged images...')
    f_types = CFG['forgery_types']
    skipped = 0
    for _, row in tqdm(forgery_pool.iterrows(), total=len(forgery_pool)):
        ftype = random.choice(f_types)
        forged_img, mask, changes = generate_forgery(
            row['image_path'], row['box_file'], ftype
        )
        if forged_img is None:
            skipped += 1
            continue

        img_dst  = os.path.join(data_dir, 'forged', f'{row["stem"]}_forged.jpg')
        mask_dst = os.path.join(data_dir, 'masks',  f'{row["stem"]}_mask.png')

        Image.fromarray(forged_img).save(img_dst, quality=92)
        cv2.imwrite(mask_dst, mask * 255)

        rows.append({
            'image_path'  : img_dst,
            'mask_path'   : mask_dst,
            'label'       : 1,
            'label_name'  : 'forged',
            'forgery_type': ftype,
            'changes'     : str(changes),
        })

    df_out = pd.DataFrame(rows)
    df_out.to_csv(os.path.join(data_dir, 'metadata.csv'), index=False)
    print(f'\n✅ Dataset built!')
    print(f'   Real    : {(df_out.label==0).sum()}')
    print(f'   Forged  : {(df_out.label==1).sum()}')
    print(f'   Skipped : {skipped}')
    return df_out


df = build_dataset(df_sroie, CFG['data_dir'], CFG['forgery_ratio'])
print(df['label_name'].value_counts())

Copying real images...


100%|██████████| 973/973 [00:01<00:00, 661.03it/s]



Generating forged images...


100%|██████████| 973/973 [10:59<00:00,  1.48it/s]


✅ Dataset built!
   Real    : 973
   Forged  : 930
   Skipped : 43
label_name
real      973
forged    930
Name: count, dtype: int64


In [9]:
from google.colab import drive
import shutil, os
from pathlib import Path
from tqdm import tqdm

drive.mount('/content/drive', force_remount=False)

DRIVE_SAVE_DIR = '/content/drive/MyDrive/receipt_forgery_dataset_new'

def is_complete(drive_dir, local_cfg_dir):
    """Check all 3 folders + metadata.csv exist and have correct file counts."""
    if not os.path.exists(os.path.join(drive_dir, 'metadata.csv')):
        return False, 'metadata.csv missing'
    for folder in ['real', 'forged', 'masks']:
        dst = os.path.join(drive_dir, folder)
        src = os.path.join(local_cfg_dir, folder)
        if not os.path.exists(dst):
            return False, f'{folder}/ folder missing'
        if os.path.exists(src):
            local_n  = len(list(Path(src).glob('*')))
            drive_n  = len(list(Path(dst).glob('*')))
            if drive_n < local_n:
                return False, f'{folder}/ incomplete ({drive_n}/{local_n} files)'
    return True, 'ok'


complete, reason = is_complete(DRIVE_SAVE_DIR, CFG['data_dir'])

if not complete:
    print(f'💾 Saving dataset to Google Drive...')
    print(f'   Reason: {reason}')
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

    for folder in ['real', 'forged', 'masks']:
        src_dir = os.path.join(CFG['data_dir'], folder)
        dst_dir = os.path.join(DRIVE_SAVE_DIR, folder)

        if not os.path.exists(src_dir):
            print(f'   ⚠️  {folder} not found locally — skipping')
            continue

        os.makedirs(dst_dir, exist_ok=True)

        # Only copy files not already on Drive (safe to re-run)
        existing = set(os.listdir(dst_dir))
        files    = [f for f in Path(src_dir).glob('*') if f.name not in existing]

        if not files:
            print(f'   ✅ {folder:8s} — already complete, skipped')
            continue

        print(f'\n   Copying {folder} ({len(files)} new files)...')
        failed = 0
        for fp in tqdm(files, desc=f'  {folder}', leave=False):
            try:
                shutil.copy2(str(fp), os.path.join(dst_dir, fp.name))
            except Exception as e:
                failed += 1

        saved = len(list(Path(dst_dir).glob('*')))
        print(f'   ✅ {folder:8s} → {saved} files on Drive'
              + (f'  ({failed} failed)' if failed else ''))

    # Save metadata.csv
    shutil.copy(
        os.path.join(CFG['data_dir'], 'metadata.csv'),
        os.path.join(DRIVE_SAVE_DIR, 'metadata.csv')
    )
    print('\n✅ metadata.csv saved')

    total = sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files_list in os.walk(DRIVE_SAVE_DIR)
        for f in files_list
    )
    print(f'\n✅ Dataset saved to : {DRIVE_SAVE_DIR}')
    print(f'   Total size       : {total/1e6:.1f} MB')

else:
    print(f'✅ Dataset complete on Drive — skipped.')
    print(f'   Location : {DRIVE_SAVE_DIR}')
    total = sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files_list in os.walk(DRIVE_SAVE_DIR)
        for f in files_list
    )
    print(f'   Total size: {total/1e6:.1f} MB')

Mounted at /content/drive
💾 Saving dataset to Google Drive...
   Reason: metadata.csv missing

   Copying real (973 new files)...


   ✅ real     → 973 files on Drive

   Copying forged (930 new files)...


   ✅ forged   → 930 files on Drive

   Copying masks (930 new files)...


   ✅ masks    → 930 files on Drive

✅ metadata.csv saved

✅ Dataset saved to : /content/drive/MyDrive/receipt_forgery_dataset_new
   Total size       : 1040.5 MB


# END